In [2]:
# %%
import uproot
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier, Pool
from catboost.utils import get_roc_curve
import sklearn.metrics as metrics

# %% [markdown]
# Define constants and parameters

TOTAL_KEYS = [
    'Jpsi_PT', 'L_END_VRHO', 'L_BPVDIRA', 'L_BPVIP', 'L_BPVIPCHI2',
    'Lb_BPVDIRA', 'Lb_BPVIP', 'Lb_BPVVDRHO', 'Lb_MAXDOCA', 'Lb_P', 'Lb_PT',
    'Lb_CHI2', 'p_PID_P', 'p_MINIP', 'mup_PID_MU', 'L_PT', 'Jpsi_MAXDOCA',
    'Lb_MINIPCHI2', 'L_P', 'Jpsi_BPVDIRA', 'L_CHI2', 'Lb_ETA', 'L_END_VZ',
    'L_END_VX', 'L_END_VY', 'Jpsi_ETA', 'p_P', 'p_PT', 'p_ETA',
    'p_GHOSTPROB', 'pim_P', 'pim_PT', 'pim_ETA', 'pim_GHOSTPROB', 'L_MASS'
]

PRESELECTION_KEYS = ['SUMCHI2', 'Cos_xi_LbP', 'Cos_xi_LP', 'xi_LbP', 'xi_LP']

TRAINING_VARIABLES = TOTAL_KEYS + PRESELECTION_KEYS

PRESELECTION_CUT = '(Jpsi_P > 10000) & (mup_PID_MU > 0) & (mum_PID_MU > 0) & (p_PID_P > 0) & (Jpsi_MAXDOCA < 0.15) & (p_P > 10000) & (p_PT > 400) & (pim_P > 2000) & (L_PT > 450) & (Lb_CHI2 < 150) & (Jpsi_CHI2 < 4) & (L_CHI2 < 100)'
SECOND_CUTS = '(p_PID_P > 0)'

MC_FILES = [
    "/l/izaac/data/24mc/jpsiLambda/mu/00225128_00000001_1.dvtuple.root",
    "/l/izaac/data/24mc/jpsiLambda/mu/00225128_00000002_1.dvtuple.root",
    "/l/izaac/data/24mc/jpsiLambda/md/00225126_00000001_1.dvtuple.root",
    "/l/izaac/data/24mc/jpsiLambda/md/00225126_00000002_1.dvtuple.root"
]

# %% [markdown]
# Function Definitions

def inspect_root_file(file):
    with uproot.open(file) as f:
        print(f.keys())

def load_data(files, total_keys):
    data_list = []
    for file in files:
        with uproot.open(file) as f:
            print(f.keys())  # Print available keys to inspect
            tree = f["Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree"]  # Update this key based on inspection
            data = tree.arrays(total_keys + ['Lb_BKGCAT', 'p_TRUEORIGIN_VZ', 'Jpsi_CHI2', 'p_PX', 'p_PY', 'p_PZ', 'L_PX', 'L_PY', 'L_PZ', 'Lb_PX', 'Lb_PY', 'Lb_PZ', 'Lb_MASS', 'EVENTNUMBER', 'Jpsi_P', 'mum_PID_MU'], library="pd")
            data_list.append(data)
    return pd.concat(data_list, axis=0)

def apply_preselection(data, cut):
    return data.query(cut)

def compute_new_variables(data):
    data['Cos_xi_LP'] = (data['p_PX']*data['L_PX'] + data['p_PY']*data['L_PY'] + data['p_PZ']*data['L_PZ']) / (data['p_P'] * data['L_P'])
    data['Cos_xi_LbP'] = (data['p_PX']*data['Lb_PX'] + data['p_PY']*data['Lb_PY'] + data['p_PZ']*data['Lb_PZ']) / (data['p_P'] * data['Lb_P'])
    data['xi_LP'] = np.arccos(data['Cos_xi_LP'])
    data['xi_LbP'] = np.arccos(data['Cos_xi_LbP'])
    data['SUMCHI2'] = data['L_CHI2'] + data['Lb_CHI2'] + data['Jpsi_CHI2']
    return data

def prepare_training_data(signal_data, background_data, data24):
    # Filter and merge data
    signal_data_filtered = signal_data.query(PRESELECTION_CUT)
    background_data_filtered = background_data.query(PRESELECTION_CUT)
    
    # Compute new variables
    signal_data_filtered = compute_new_variables(signal_data_filtered)
    background_data_filtered = compute_new_variables(background_data_filtered)
    
    # Combine data
    combined_data = pd.concat([signal_data_filtered, background_data_filtered, data24])
    combined_data = combined_data.dropna()
    
    return combined_data

def plot_roc_curve(model, X_eval, y_eval):
    probs = model.predict_proba(X_eval)[:,1]
    fpr, tpr, _ = metrics.roc_curve(y_eval, probs)
    roc_auc = metrics.auc(fpr, tpr)

    plt.figure(figsize=(10,10))
    plt.title('Receiver Operating Characteristic (ROC)', fontsize=26)
    plt.plot(fpr, tpr, 'b', label=f'AUC = {roc_auc:.6f}', linewidth=2)
    plt.legend(loc='lower right', fontsize=24)
    plt.plot([0, 1], [0, 1], 'r--')
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.ylabel('True Positive Rate', fontsize=24)
    plt.xlabel('False Positive Rate', fontsize=24)
    plt.xticks(fontsize=24)
    plt.yticks(fontsize=24)
    plt.show()

def plot_feature_importance(model, feature_names):
    sorted_idx = model.feature_importances_.argsort()
    plt.figure(figsize=(10,10))
    plt.barh(feature_names[sorted_idx], model.feature_importances_[sorted_idx], color='turquoise')
    plt.xlabel("Feature Importance", fontsize=24)
    plt.xticks(fontsize=22)
    plt.yticks(fontsize=14)
    plt.show()

# %% [markdown]
# Inspect the keys in the ROOT files

for file in MC_FILES:
    inspect_root_file(file)

# %% [markdown]
# Load and process MC data

signal_data = load_data(MC_FILES[:2], TOTAL_KEYS)
background_data = load_data(MC_FILES[2:], TOTAL_KEYS)

# Apply preselection cuts
signal_data_selected = apply_preselection(signal_data, PRESELECTION_CUT)
background_data_selected = apply_preselection(background_data, PRESELECTION_CUT)

# Load and process 2024 data
data24 = pd.read_pickle('data24_magup_training.pkl')
data24 = compute_new_variables(data24)

# %% [markdown]
# Prepare training data

training_data = prepare_training_data(signal_data_selected, background_data_selected, data24)

# %% [markdown]
# Train model

X = training_data.drop(columns=["SIGNAL", "Lb_BKGCAT", "p_TRUEORIGIN_VZ", 'CLASS','Lb_MASS','Cos_xi_LbP','Cos_xi_LP', 'L_MASS'])
y = training_data['SIGNAL']

X_train, X_eval, y_train, y_eval = train_test_split(X, y, test_size=0.2, random_state=69)

train_pool = Pool(X_train, y_train, feature_names=list(X_train.columns))
eval_pool = Pool(X_eval, y_eval, feature_names=list(X_eval.columns))

model_params = {
    'iterations': 1000, 
    'loss_function': 'Logloss',
    'train_dir': 'crossentropy',
    'allow_writing_files': False,
    'od_type': 'IncToDec',
    'auto_class_weights': 'Balanced',
    'random_seed': 69
}

model = CatBoostClassifier(**model_params, custom_metric=['Logloss', 'AUC:hints=skip_train~false', 'Accuracy'])
model.fit(train_pool, eval_set=eval_pool, verbose=True, plot=False)

# %% [markdown]
# Plot ROC curve

plot_roc_curve(model, X_eval, y_eval)

# %% [markdown]
# Plot feature importance

plot_feature_importance(model, X.columns)


['Hlt2BandQ_Lb2JpsiLambdaTT;1', 'Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1']
['Hlt2BandQ_Lb2JpsiLambdaTT;1', 'Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1']
['Hlt2BandQ_Lb2JpsiLambdaTT;1', 'Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1']
['Hlt2BandQ_Lb2JpsiLambdaTT;1', 'Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1']
['Hlt2BandQ_Lb2JpsiLambdaTT;1', 'Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1']
['Hlt2BandQ_Lb2JpsiLambdaTT;1', 'Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1']
['Hlt2BandQ_Lb2JpsiLambdaTT;1', 'Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1']
['Hlt2BandQ_Lb2JpsiLambdaTT;1', 'Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1']


/l/javeser/micromamba/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
/l/javeser/micromamba/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
/l/javeser/micromamba/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
/l/javeser/micromamba/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)


KeyError: "['SIGNAL', 'CLASS'] not found in axis"